# 02 · OCR Pipeline — Ön İşleme + Tesseract

Bu notebook:
1. Görüntü ön işleme: gürültü temizleme → büyütme → **eğiklik düzeltme** → Otsu binarizasyon
2. Tesseract ile OCR: **metin + kelime kutuları** (LayoutLMv3 formatında, 0–1000 normalize)
3. Tüm sonuçları belge başına JSON olarak Drive'a **önbellekler** (yarıda kalırsa kaldığı yerden devam eder)
4. `ocr_dataset.csv` üretir → sınıflandırma notebook'larının girdisi
5. Ön işlemenin etkisini **CER / WER** ile ölçer (rapor için)

**Neden Tesseract?** Kelime düzeyinde kutu + güven skoru verir, kurulumu kararlıdır ve CPU'da çalışır (GPU kotası eğitime kalır). PaddleOCR/EasyOCR ile karşılaştırma bonus olarak eklenebilir.

> Önce `01_data_preparation.ipynb` çalışmış olmalı.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract jiwer opencv-python-headless
!tesseract --version | head -1

## 2. Drive ve yollar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/bitirme_ocr")   # tüm proje çıktıları burada
DATA = ROOT / "data"
IMG_DIR = DATA / "images"
OCR_DIR = DATA / "ocr"
for p in (IMG_DIR, OCR_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Projedeki 4 sınıf (sıra = model etiket id'leri 0..3)
CLASSES = ["email", "form", "invoice", "resume"]
print("Proje klasörü:", ROOT)

## 3. OCR çekirdek modülü

Modül Drive'a `ocr_core.py` olarak yazılır; sonraki notebook'lar ve Gradio demosu aynı fonksiyonları `import` eder.

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


In [ ]:
import sys, importlib
sys.path.insert(0, str(ROOT))
import ocr_core
importlib.reload(ocr_core)
from ocr_core import preprocess, run_ocr

### OCR ayarı (pilot deney sonucu)

03'teki pilot deneyde (40 validation belgesi) **ön işlemesiz ham görüntü** en iyi sonucu verdi:

| Ayar | Ortalama güven | Sözlükte bulunan kelime oranı |
|---|---|---|
| A · ham görüntü | **65,9** | **0,591** |
| B · median + büyütme | 44,9 | 0,448 |
| C · B + eğiklik düzeltme | 45,5 | 0,460 |
| D · C + Otsu (v1) | 38,3 | 0,405 |

Median filtre ve büyütme, düşük çözünürlüklü taramalardaki ince harf çizgilerini bozuyor; Tesseract zaten kendi ikilileştirmesini yapıyor.
Bu nedenle v2'de görüntü doğrudan Tesseract'a verilir. v1 sonuçları `data/ocr/` klasöründe karşılaştırma için saklanır.

In [ ]:
USE_PREPROCESS = False   # v1: True (median + büyütme + eğiklik + Otsu) · v2: False (ham görüntü)
OCR_DIR = DATA / ("ocr" if USE_PREPROCESS else "ocr_raw")
OCR_DIR.mkdir(parents=True, exist_ok=True)
print("OCR sonuçları:", OCR_DIR)

## 4. Ön işleme: önce / sonra

In [ ]:
import cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

meta = pd.read_csv(DATA / "metadata.csv")
row = meta[meta.label == "resume"].sample(1, random_state=1).iloc[0]
raw = cv2.imread(str(DATA / row["path"]), cv2.IMREAD_GRAYSCALE)
pre, angle = preprocess(raw)

fig, ax = plt.subplots(1, 2, figsize=(14, 9))
ax[0].imshow(raw, cmap="gray"); ax[0].set_title(f"Orijinal {raw.shape[::-1]}")
ax[1].imshow(pre, cmap="gray"); ax[1].set_title(f"Ön işlenmiş {pre.shape[::-1]} · eğiklik {angle:.2f}°")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(ROOT / "on_isleme_ornek.png", dpi=120); plt.show()

## 5. Tek belgede OCR denemesi

In [ ]:
res = run_ocr(pre if USE_PREPROCESS else raw)
print(f"Kelime: {res['n_words']}  ·  Ortalama güven: {res['mean_conf']:.1f}")
print("-" * 60)
print(res["text"][:1500])

In [ ]:
# Kelime kutularını çiz (LayoutLMv3'e verilecek kutular)
shown = pre if USE_PREPROCESS else raw
vis = cv2.cvtColor(shown, cv2.COLOR_GRAY2BGR)
h, w = shown.shape
for (x0, y0, x1, y1), c in zip(res["boxes"], res["confs"]):
    color = (0, 170, 0) if c >= 60 else (0, 0, 255)   # düşük güven = kırmızı
    cv2.rectangle(vis, (x0 * w // 1000, y0 * h // 1000), (x1 * w // 1000, y1 * h // 1000), color, 2)
plt.figure(figsize=(9, 12)); plt.imshow(vis[..., ::-1]); plt.axis("off"); plt.show()

## 6. Tüm veri setine OCR uygula

- Her belge için `data/ocr/<split>/<sınıf>/<dosya>.json` yazılır
- Var olan JSON'lar atlanır → oturum düşerse hücreyi tekrar çalıştırmanız yeterli
- Tesseract ayrı süreç olarak çalıştığı için thread havuzu ile paralelleştirilir
- Süre: Colab CPU'da 2.400 belge için yaklaşık 30–60 dk (v2 ham görüntü daha küçük olduğu için daha hızlı)

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def ocr_path(img_rel):
    return OCR_DIR / Path(img_rel).with_suffix(".json").relative_to("images")

def process_one(row):
    out = ocr_path(row["path"])
    if out.exists():
        return "skip"
    img = cv2.imread(str(DATA / row["path"]), cv2.IMREAD_GRAYSCALE)
    pre, angle = preprocess(img) if USE_PREPROCESS else (img, 0.0)
    res = run_ocr(pre)
    res["skew_angle"] = angle
    res["preprocess"] = USE_PREPROCESS
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(res, ensure_ascii=False))
    return "ok"

workers = max(2, os.cpu_count() or 2)
todo = meta.to_dict("records")
stats = {"ok": 0, "skip": 0, "error": 0}
with ThreadPoolExecutor(max_workers=workers) as pool:
    futs = {pool.submit(process_one, r): r for r in todo}
    for f in tqdm(as_completed(futs), total=len(futs)):
        try:
            stats[f.result()] += 1
        except Exception as e:
            stats["error"] += 1
            print("HATA:", futs[f]["path"], e)
print(stats)

## 7. `ocr_dataset.csv` oluştur

In [ ]:
records = []
for r in meta.to_dict("records"):
    p = ocr_path(r["path"])
    if not p.exists():
        continue
    o = json.loads(p.read_text())
    records.append({**r, "ocr_json": str(p.relative_to(DATA)), "text": o["text"],
                    "n_words": o["n_words"], "mean_conf": o["mean_conf"],
                    "skew_angle": o.get("skew_angle", 0.0)})

ocr_df = pd.DataFrame(records)
ocr_df.to_csv(DATA / "ocr_dataset.csv", index=False)
print(ocr_df.shape)
ocr_df[["path", "split", "label", "n_words", "mean_conf"]].head()

## 8. OCR istatistikleri (rapor için)

In [ ]:
summary = ocr_df.groupby("label").agg(
    belge=("path", "count"),
    ort_kelime=("n_words", "mean"),
    medyan_kelime=("n_words", "median"),
    ort_guven=("mean_conf", "mean"),
    bos_metin=("n_words", lambda s: int((s < 5).sum())),
).round(1)
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ocr_df.boxplot(column="n_words", by="label", ax=ax[0], showfliers=False)
ax[0].set_title("Belge başına kelime sayısı"); ax[0].set_xlabel("")
ocr_df.boxplot(column="mean_conf", by="label", ax=ax[1], showfliers=False)
ax[1].set_title("Ortalama OCR güveni"); ax[1].set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.savefig(ROOT / "ocr_istatistik.png", dpi=120); plt.show()

## 9. OCR doğruluğu: CER / WER (ön işleme etkisi)

RVL-CDIP'te metin etiketi yok. Bu yüzden **küçük bir doğrulama seti** hazırlayın:

1. `data/ground_truth/` klasörüne 10–20 belge seçin (her sınıftan birkaç tane)
2. Her biri için görüntü adıyla aynı isimde `.txt` dosyasına metni elle yazın
   (örnek: `test_resume_00003.txt`)
3. Aşağıdaki hücre **ön işlemesiz** ve **ön işlemeli** OCR'ı karşılaştırır

Bu tablo raporda "ön işlemenin katkısı" bölümü için doğrudan kullanılabilir.

In [ ]:
import jiwer

GT_DIR = DATA / "ground_truth"
GT_DIR.mkdir(exist_ok=True)
norm = lambda s: " ".join(s.lower().split())
by_name = {Path(p).stem: p for p in meta["path"]}

rows = []
for gt_file in sorted(GT_DIR.glob("*.txt")):
    rel = by_name.get(gt_file.stem)
    if rel is None:
        print("Eşleşen görüntü yok:", gt_file.name); continue
    gt = norm(gt_file.read_text())
    img = cv2.imread(str(DATA / rel), cv2.IMREAD_GRAYSCALE)
    raw_txt = norm(run_ocr(img)["text"])
    pre_txt = norm(run_ocr(preprocess(img)[0])["text"])
    rows.append({"belge": gt_file.stem,
                 "CER_ham": jiwer.cer(gt, raw_txt), "CER_on_isleme": jiwer.cer(gt, pre_txt),
                 "WER_ham": jiwer.wer(gt, raw_txt), "WER_on_isleme": jiwer.wer(gt, pre_txt)})

if rows:
    cer_df = pd.DataFrame(rows).set_index("belge")
    display(cer_df.round(3))
    print("Ortalama:"); display(cer_df.mean().round(3))
else:
    print(f"{GT_DIR} içine .txt dosyaları ekleyip tekrar çalıştırın.")

## Çıktılar

| Dosya | Kullanım |
|---|---|
| `data/ocr_raw/**.json` | v2 OCR sonuçları (kullanılan) · `data/ocr/` = v1 (karşılaştırma için) |
| `data/ocr_dataset.csv` | Metin + etiket → TF-IDF/SVM (03) |
| `ocr_core.py` | Diğer notebook'lar ve Gradio demosu |
| `*.png` grafikler | Rapor görselleri |

Sonraki adım → **03_baseline_tfidf_svm.ipynb**